# Scorecard 300-850

Transformamos la probabilidad de default en un score crediticio estilo FICO.

**Semáforo de riesgo:**
- 🟢 Verde > 650: riesgo bajo
- 🟡 Amarillo 501-650: riesgo medio
- 🔴 Rojo ≤ 500: riesgo alto

In [ ]:
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.insert(0, '..')

from src.data.preprocess import dividir_datos
from src.models.evaluate import probabilidad_a_score

modelo = joblib.load('models/best_model.pkl')
df = pd.read_parquet('data/processed/dataset_features.parquet')
X_train, X_test, y_train, y_test = dividir_datos(df)

probas = modelo.predict_proba(X_test)[:, 1]
scores = [probabilidad_a_score(p) for p in probas]
df_scores = pd.DataFrame({
    'score': scores,
    'default': y_test.values,
    'proba': probas
})
print(f"Score mínimo: {min(scores)} | máximo: {max(scores)}")
print(f"Score mediano: {np.median(scores):.0f}")

## Distribución de scores por default

In [ ]:
fig = px.histogram(
    df_scores, x='score',
    color=df_scores['default'].astype(str),
    title='Distribución del Score Crediticio (300-850) por Default',
    barmode='overlay', opacity=0.7, nbins=50,
    color_discrete_map={'0': '#2196F3', '1': '#F44336'},
    labels={'score': 'Score crediticio', 'color': 'Default'}
)
fig.add_vline(x=500, line_dash='dash', line_color='red', annotation_text='Rojo ≤500')
fig.add_vline(x=650, line_dash='dash', line_color='green', annotation_text='Verde >650')
fig.show()

## Tasa de mora por decil de score

In [ ]:
df_scores['decil'] = pd.qcut(df_scores['score'], q=10, labels=False) + 1
resumen_decil = df_scores.groupby('decil').agg(
    score_min=('score', 'min'),
    score_max=('score', 'max'),
    tasa_mora=('default', 'mean'),
    registros=('default', 'count')
).reset_index()
resumen_decil['tasa_mora_pct'] = (resumen_decil['tasa_mora'] * 100).round(2)
print(resumen_decil.to_string(index=False))

fig = px.bar(resumen_decil, x='decil', y='tasa_mora',
             title='Tasa de Mora por Decil de Score (1=peor, 10=mejor)',
             color='tasa_mora', color_continuous_scale='RdYlGn_r',
             labels={'tasa_mora': 'Tasa de mora', 'decil': 'Decil'})
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

## Tasa de aprobación vs tasa de mora por semáforo

In [ ]:
for zona, label in [(df_scores['score'] > 650, '🟢 Verde (>650)'),
                       ((df_scores['score'] > 500) & (df_scores['score'] <= 650), '🟡 Amarillo (501-650)'),
                       (df_scores['score'] <= 500, '🔴 Rojo (≤500)')]:
    subset = df_scores[zona]
    print(f"{label}: {len(subset):,} solicitudes ({len(subset)/len(df_scores):.1%}) | "
          f"Mora: {subset['default'].mean():.1%}")